# Contested Norms Study
## Data Prep: Stratified sampling of accounts

Created: 2025-10-13  
Updated: 2025-10-20  
Authors: Edward L. Platt  

In [ ]:
%matplotlib inline
from configparser import ConfigParser
import csv
from datetime import datetime, timedelta
import json
import logging
import math
import os
import pytz
import random
import simplejson as json
import sys
from tqdm import tqdm

utc=pytz.UTC

### LOAD CONFIGURATION
config = ConfigParser()
config.read('config.ini')
config.write(sys.stdout)

subreddit_id = config.get("Source", "subreddit_id")
start_time = utc.localize(datetime.strptime(config.get("Source", "start_time"), "%Y-%m-%d %H:%M:%S"))
end_time = utc.localize(datetime.strptime(config.get("Source", "end_time"), "%Y-%m-%d %H:%M:%S"))

account_file = config.get("Transformed", "account_file")

seed = config.getint("Sample", "seed")
post_activity_min = config.getint("Sample", "post_activity_min")
comment_activity_min = config.getint("Sample", "comment_activity_min")
strata_size = config.getint("Sample", "strata_size")

script_name = "contested_norms-{}-sample".format(subreddit_id)
script_date = datetime.now().strftime('%Y-%m-%d')

# Configure logging
logging.basicConfig(
    filename='{}-{}.log'.format(script_date, script_name),
    format='%(asctime)s:%(levelname)s:%(message)s',
    level=logging.DEBUG)
logger = logging.getLogger("CivilServant-Analysis")
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.DEBUG)
formatter = logging.Formatter('%(asctime)s:%(levelname)s:%(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)

### LOAD ANALYSIS CODE
from pipeline import DataFile
from pipeline.transforms import CivilServantTransformToAccounts

In [ ]:
transform = CivilServantTransformToAccounts(
    start_time, end_time, subreddit_id)
accounts = DataFile(transform, directory="transformed", filename=account_file)

In [ ]:
logger.info("Identifying active posters and commenters")
active_both = []
active_posters = []
active_commenters = []
inactive = 0
for row in accounts.rows():
    poster = row['num.posts'] >= post_activity_min
    commenter = row['num.comments'] >= comment_activity_min
    if poster and commenter:
        active_both.append(row)
    else:
        if poster:
            active_posters.append(row)
        elif commenter:
            active_commenters.append(row)
        else:
            inactive += 1
logger.info("  Done")
logger.info("  {} poster non-commenters".format(len(active_posters)))
logger.info("  {} commenter non-posters".format(len(active_commenters)))
logger.info("  {} poster commenters".format(len(active_both)))
logger.info("  {} inactive".format(inactive))

In [ ]:
logger.info("Sampling")
logger.info("  Seeding random number generator: {}".format(seed))
random.seed(seed)
poster_sample = random.sample(active_posters, strata_size)
commenter_sample = random.sample(active_commenters, strata_size)
both_sample = random.sample(active_both, strata_size)
logger.info("  Done")
logger.info("  Sampled {} poster non-commenters".format(len(poster_sample)))
logger.info("  Sampled {} commenter non-posters".format(len(commenter_sample)))
logger.info("  Sampled {} poster commenters".format(len(both_sample)))

In [ ]:
filename = "transformed/2025-10-20-2ssp3-sample-accounts.tsv"
logger.info("Writing sample to {}".format(filename))
sample = poster_sample + commenter_sample + both_sample
with open(filename, 'w', encoding='utf8') as f:
    writer = csv.DictWriter(f, fieldnames=list(poster_sample[0].keys()), delimiter='\t')
    writer.writeheader()
    for row in sample:
        writer.writerow(row)
logger.info("  Done")    